#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

import joblib

#### Loading the Prepared Datasets

In [2]:
# Loading the saved training and testing datasets

processed_dir = Path("../data/processed")

X_train = pd.read_parquet(processed_dir / "X_train_raw.parquet")
X_test = pd.read_parquet(processed_dir / "X_test_raw.parquet")

y_train = pd.read_parquet(
    processed_dir / "y_train.parquet"
)["severity"]

y_test = pd.read_parquet(
    processed_dir / "y_test.parquet"
)["severity"]

# Displaying dataset dimensions

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("\nTraining Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

# Displaying feature data types

print("\nTraining Feature Data Types:")
display(X_train.dtypes.to_frame(name="Data Type"))

# Displaying sample records

print("\nTraining Dataset Preview:")
display(X_train.head())

Training Features Shape: (800, 4)
Testing Features Shape: (200, 4)

Training Target Shape: (800,)
Testing Target Shape: (200,)

Training Feature Data Types:


,Data Type
defect_type,str
defect_date,datetime64[us]
defect_location,str
inspection_method,str



Training Dataset Preview:


,defect_type,defect_date,defect_location,inspection_method
688,Cosmetic,2024-03-09,Surface,Manual Testing
936,Structural,2024-04-18,Internal,Manual Testing
820,Cosmetic,2024-02-11,Surface,Visual Inspection
708,Functional,2024-03-10,Component,Automated Testing
565,Cosmetic,2024-02-17,Internal,Automated Testing


#### Engineering Temporal Features

In [3]:
# Engineering temporal features from defect dates

for dataset in [X_train, X_test]:

    dataset["Month"] = dataset["defect_date"].dt.month
    dataset["Day"] = dataset["defect_date"].dt.day
    dataset["Day_of_Week"] = dataset["defect_date"].dt.dayofweek
    dataset["Is_Weekend"] = (
        dataset["Day_of_Week"] >= 5
    ).astype(int)

# Removing the original date variable

X_train = X_train.drop(columns=["defect_date"])
X_test = X_test.drop(columns=["defect_date"])

# Displaying the engineered datasets

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("\nEngineered Feature Names:")
print(X_train.columns.tolist())

print("\nTraining Dataset Preview:")
display(X_train.head())

print("\nMissing Values After Feature Engineering:")
display(X_train.isnull().sum().to_frame(name="Missing Values"))

Training Features Shape: (800, 7)
Testing Features Shape: (200, 7)

Engineered Feature Names:
['defect_type', 'defect_location', 'inspection_method', 'Month', 'Day', 'Day_of_Week', 'Is_Weekend']

Training Dataset Preview:


,defect_type,defect_location,inspection_method,Month,Day,Day_of_Week,Is_Weekend
688,Cosmetic,Surface,Manual Testing,3,9,5,1
936,Structural,Internal,Manual Testing,4,18,3,0
820,Cosmetic,Surface,Visual Inspection,2,11,6,1
708,Functional,Component,Automated Testing,3,10,6,1
565,Cosmetic,Internal,Automated Testing,2,17,5,1



Missing Values After Feature Engineering:


,Missing Values
defect_type,0
defect_location,0
inspection_method,0
Month,0
Day,0
Day_of_Week,0
Is_Weekend,0


#### Observations on Temporal Feature Engineering

- Four temporal features were extracted from `defect_date`: `Month`, `Day`, `Day_of_Week`, and `Is_Weekend`.
- The feature count increased from **4 to 7**, consisting of 3 categorical variables and 4 engineered temporal variables.
- The original `defect_date` column was removed after extracting the relevant calendar information.
- The `Year` feature was excluded because all records belong to 2024, making it a constant variable with no discriminatory value.
- The `Is_Weekend` feature provides a binary representation of weekday and weekend defect occurrences.
- No missing values were introduced during feature engineering, and the training and testing datasets retain their original dimensions of **800 and 200 records**, respectively.

#### Identifying Features for Preprocessing

In [4]:
# Identifying categorical, numerical, and binary features

categorical_features = [
    "defect_type",
    "defect_location",
    "inspection_method"
]

numerical_features = [
    "Month",
    "Day",
    "Day_of_Week"
]

binary_features = [
    "Is_Weekend"
]

# Displaying feature groups

print("Categorical Features:", categorical_features)
print("Numerical Features:", numerical_features)
print("Binary Features:", binary_features)

print("\nTotal Original Features:", X_train.shape[1])

# Validating feature assignments

all_features = (
    categorical_features
    + numerical_features
    + binary_features
)

print("\nAll Features Accounted For:",
      set(all_features) == set(X_train.columns))

print("Duplicate Feature Assignments:",
      len(all_features) - len(set(all_features)))

print("\nTraining Feature Data Types:")
display(X_train.dtypes.to_frame(name="Data Type"))

Categorical Features: ['defect_type', 'defect_location', 'inspection_method']
Numerical Features: ['Month', 'Day', 'Day_of_Week']
Binary Features: ['Is_Weekend']

Total Original Features: 7

All Features Accounted For: True
Duplicate Feature Assignments: 0

Training Feature Data Types:


,Data Type
defect_type,str
defect_location,str
inspection_method,str
Month,int32
Day,int32
Day_of_Week,int32
Is_Weekend,int64


#### Observations on Feature Identification

- The feature-engineered dataset contains **7 predictor variables**, comprising 3 categorical, 3 numerical, and 1 binary feature.
- The categorical variables are `defect_type`, `defect_location`, and `inspection_method`, each requiring encoding for machine learning compatibility.
- The numerical features `Month`, `Day`, and `Day_of_Week` are identified for standardization.
- The binary feature `Is_Weekend` is already represented using 0 and 1, eliminating the need for additional encoding or scaling.
- Feature validation confirms that all 7 variables are assigned to preprocessing groups, with no missing or duplicate assignments.
- The feature data types are consistent with their intended transformations.

#### Building and Applying the Preprocessing Pipeline

In [6]:
# Defining the preprocessing pipeline

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "binary",
            "passthrough",
            binary_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

# Fitting the preprocessing pipeline on training data

X_train_transformed = preprocessor.fit_transform(X_train)

# Applying the fitted pipeline to testing data

X_test_transformed = preprocessor.transform(X_test)

# Retrieving transformed feature names

feature_names = preprocessor.get_feature_names_out()

# Converting transformed arrays into DataFrames

X_train_processed = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=X_test.index
)

# Displaying transformed dataset dimensions

print("Processed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape:", X_test_processed.shape)

print("\nTotal Processed Features:", len(feature_names))

print("\nProcessed Feature Names:")
print(feature_names.tolist())

print("\nProcessed Training Dataset Preview:")
display(X_train_processed.head())

Processed Training Shape: (800, 13)
Processed Testing Shape: (200, 13)

Total Processed Features: 13

Processed Feature Names:
['categorical__defect_type_Cosmetic', 'categorical__defect_type_Functional', 'categorical__defect_type_Structural', 'categorical__defect_location_Component', 'categorical__defect_location_Internal', 'categorical__defect_location_Surface', 'categorical__inspection_method_Automated Testing', 'categorical__inspection_method_Manual Testing', 'categorical__inspection_method_Visual Inspection', 'numerical__Month', 'numerical__Day', 'numerical__Day_of_Week', 'binary__Is_Weekend']

Processed Training Dataset Preview:


,categorical__defect_type_Cosmetic,categorical__defect_type_Functional,categorical__defect_type_Structural,categorical__defect_location_Component,categorical__defect_location_Internal,categorical__defect_location_Surface,categorical__inspection_method_Automated Testing,categorical__inspection_method_Manual Testing,categorical__inspection_method_Visual Inspection,numerical__Month,numerical__Day,numerical__Day_of_Week,binary__Is_Weekend
688,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,-0.241302,-0.713724,0.902656,1.0
936,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.345451,0.310760,-0.090638,0.0
820,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.828055,-0.486061,1.399303,1.0
708,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,-0.241302,-0.599892,1.399303,1.0
565,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,-0.828055,0.196929,0.902656,1.0


#### Observations on Processed Data Validation

- The processed training and testing datasets contain **800 × 13** and **200 × 13** records and features, respectively.
- No missing values, infinite values, or duplicate feature names were identified.
- Feature consistency and target alignment checks passed, confirming that both datasets are ready for model development.

#### Validating Processed Datasets

In [7]:
# Validating processed training and testing datasets

print("Training Dataset Shape:", X_train_processed.shape)
print("Testing Dataset Shape:", X_test_processed.shape)

# Checking missing values

print("\nMissing Values:")
print("Training:", X_train_processed.isnull().sum().sum())
print("Testing:", X_test_processed.isnull().sum().sum())

# Checking infinite values

print("\nInfinite Values:")
print("Training:", np.isinf(X_train_processed.to_numpy()).sum())
print("Testing:", np.isinf(X_test_processed.to_numpy()).sum())

# Checking feature consistency

print("\nFeature Consistency:")
print(
    "Matching Feature Names:",
    X_train_processed.columns.equals(X_test_processed.columns)
)

print(
    "Matching Feature Counts:",
    X_train_processed.shape[1] == X_test_processed.shape[1]
)

# Checking target alignment

print("\nTarget Alignment:")
print(
    "Training Indices Aligned:",
    X_train_processed.index.equals(y_train.index)
)

print(
    "Testing Indices Aligned:",
    X_test_processed.index.equals(y_test.index)
)

# Checking duplicate feature names

print("\nDuplicate Feature Names:")
print(X_train_processed.columns.duplicated().sum())

Training Dataset Shape: (800, 13)
Testing Dataset Shape: (200, 13)

Missing Values:
Training: 0
Testing: 0

Infinite Values:
Training: 0
Testing: 0

Feature Consistency:
Matching Feature Names: True
Matching Feature Counts: True

Target Alignment:
Training Indices Aligned: True
Testing Indices Aligned: True

Duplicate Feature Names:
0


#### Saving Processed Datasets and Preprocessing Artifacts

In [8]:
# Defining output directories

processed_dir = Path("../data/processed")
preprocessor_dir = Path("../models/preprocessed")

processed_dir.mkdir(parents=True, exist_ok=True)
preprocessor_dir.mkdir(parents=True, exist_ok=True)

# Saving processed training and testing features

X_train_processed.to_parquet(
    processed_dir / "X_train_processed.parquet"
)

X_test_processed.to_parquet(
    processed_dir / "X_test_processed.parquet"
)

# Saving the fitted preprocessing pipeline

joblib.dump(
    preprocessor,
    preprocessor_dir / "preprocessing_pipeline.joblib"
)

# Saving transformed feature names

joblib.dump(
    feature_names.tolist(),
    preprocessor_dir / "feature_names.joblib"
)

# Verifying saved files

saved_files = [
    processed_dir / "X_train_processed.parquet",
    processed_dir / "X_test_processed.parquet",
    preprocessor_dir / "preprocessing_pipeline.joblib",
    preprocessor_dir / "feature_names.joblib"
]

print("Saved Artifacts:")

for file in saved_files:
    print(
        f"{file.name}:",
        "Saved Successfully" if file.is_file() else "Missing"
    )

Saved Artifacts:
X_train_processed.parquet: Saved Successfully
X_test_processed.parquet: Saved Successfully
preprocessing_pipeline.joblib: Saved Successfully
feature_names.joblib: Saved Successfully


#### Observations on Feature Engineering

- Temporal feature engineering and categorical encoding transformed **4 original predictors into 13 model-ready features**.
- The processed training and testing datasets contain **800 and 200 records**, respectively, with no missing or infinite values.
- The processed datasets and fitted preprocessing pipeline were saved for reuse in model development, evaluation, and explainability.